# Step 0.3 — Crop masks from MIRCA2000, seuil cellule sur fraction de SURFACE (pas pixels), pixel natif sans seuil

**Raisonnement agronomique affine (discute avec l'utilisatrice, 2026-10-13)** -- voir
[[mirca-5pct-presence-threshold-dilution]] pour l'historique complet. Deux etapes anterieures
(`0.4)` = aucun seuil, `0.2)` = seuil pixel 2% + seuil cellule 2% EN NOMBRE DE PIXELS)
avaient chacune un defaut reel :

- `0.4)` (aucun seuil) : inclut trop de bruit (trace infime repetee dans presque chaque pixel
  agricole d'Europe).
- `0.2)` (seuil pixel 2%) : detruit a tort la culture diffuse-mais-reelle (ex. pomme de
  terre francaise/belge, part moyenne par pixel 0.4-1.6%, DEJA sous un seuil de 2% -- verifie avec
  de vrais hectares).

**La vraie distinction** (proposee par l'utilisatrice) : le seuil *pixel* mesure une concentration
locale qui depend du systeme agricole (rotation diversifiee vs monoculture), pas du climat -- il ne
devrait PAS exister. Le seuil *cellule* doit mesurer si la presence est representative du climat
REGIONAL (ce que voit le SVM) -- mais un comptage de PIXELS TOUCHES ne distingue pas "beaucoup de
vraie surface etalee sur beaucoup de pixels" (ex. Picardie, 4663ha/811161ha=0.57%, legitime) de
"presque rien repete dans presque chaque pixel" (ex. Loire, 169ha/843233ha=0.02%, 25x moins, bruit) --
verifie : les deux touchent ~143/144 pixels, indiscernables par comptage, mais separes par un facteur
25-60x en vraie fraction de surface.

**Regle retenue** :
- **Pixel natif (5 arc-min) : aucun seuil** -- toute trace de surface compte (`A > 0`), pas de
  seuil de concentration locale.
- **Cellule 1deg : seuil sur la fraction de SURFACE reelle** (somme de la surface de la culture dans
  la cellule / surface de terre totale de la cellule), pas un comptage de pixels. Seuil retenu :
  **0.1%** (REVISE 2026-10-13, voir [[mirca-5pct-presence-threshold-dilution]]) -- un premier choix
  a 0.5% (base sur la seule comparaison Picardie/Loire) capturait 89% de la surface europeenne totale
  mais seulement **63% de la surface francaise reelle** (12/105 cellules de la bbox France) : ce chiffre
  global etait trompeur, domine par l'Europe de l'Est ou la pomme de terre est bien plus concentree
  par cellule, masquant que la France -- le cas meme qui motive cette refonte -- restait mal captee.
  0.1% capture 92% de la surface francaise reelle (43/105 cellules) tout en gardant Loire (0.02%)
  exclu avec une marge x5.

Parallele, non-destructif : construit `mirca2000_1deg_cellareafrac_<Crop>_rf.nc`, rien d'autre touche.


In [1]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import numpy as np
import pandas as pd
import xarray as xr
import regionmask
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb
CAL_FILE = BASE / "0_Input_data" / "masks_crops" / "MIRCA_2000" / "growing_periods_listed" / "CELL_SPECIFIC_CROPPING_CALENDARS.TXT.gz"
CELL_AREA_FILE = BASE / "0_Input_data" / "masks_crops" / "MIRCA_2000" / "cell_area_grid" / "cell_area_ha.npy"
ERA5_1DEG = BASE / "1_SVM" / "0_preprocessing_masks" / "outputs_input_data_1deg"

CELL_AREA_THR = 0.0005  # 0.05% (REVISE 2026-10-13, 2e passe : 0.1% coupait en plein milieu de la Pianure Padane -- Nord Italie ne captait que 25.4% de sa surface reelle a 0.1%, 83.9% a 0.05% ; Loire reste exclu avec marge x2.5, voir [[mirca-5pct-presence-threshold-dilution]])

LAT0, LAT1, LON0, LON1 = 35, 71, -25, 44
NLAT, NLON = LAT1 - LAT0 + 1, LON1 - LON0 + 1
NCELL = NLAT * NLON
LAND1_THR = 0.25
lat1 = np.arange(LAT0, LAT1 + 1); lon1 = np.arange(LON0, LON1 + 1)
print(f"1deg grid: {NLAT} x {NLON} = {NCELL} cells, CELL_AREA_THR={CELL_AREA_THR*100}%")

CROP_MAP = {
    'Maize_rf':        ('Maize',  'rf', None,     'mirca2000_1deg_Maize_rf.nc'),
    'Wheat-spring_rf': ('Wheat',  'rf', 'spring', 'mirca2000_1deg_Wheat-spring_rf.nc'),
    'Wheat-winter_rf': ('Wheat',  'rf', 'winter', 'mirca2000_1deg_Wheat-winter_rf.nc'),
    'Potato_rf':       ('Potato', 'rf', None,     'mirca2000_1deg_Potato_rf.nc'),
}


1deg grid: 37 x 70 = 2590 cells, CELL_AREA_THR=0.05%


C:\Users\elisa\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Grille cible + masque terre (meme source que 0.1/0.4/0.2)

In [2]:
sample_file = ERA5_1DEG / "ERA5_1deg_tasmax_1981_2015_daily.nc"
ds_grid = xr.open_dataset(sample_file)
land_mask = np.isfinite(ds_grid['tasmax'].isel(time=0).values)
ds_grid.close()
print(f"Land 1deg cells: {int(land_mask.sum())} / {land_mask.size}")


Land 1deg cells: 1332 / 2590


## 2. MIRCA2000 presence a 1deg, pixel sans seuil / cellule = fraction de surface > 0.5%

In [3]:
NROW, NCOL = 2160, 4320
ROWS_, COLS_ = np.arange(NROW), np.arange(NCOL)
lat5 = 90 - (ROWS_ + 0.5) / 12
lon5 = -180 + (COLS_ + 0.5) / 12
lat5_i = np.floor(lat5 + 0.5).astype(int)
lon5_i = np.floor(lon5 + 0.5).astype(int)
R5 = np.where((lat5_i >= LAT0) & (lat5_i <= LAT1))[0]
C5 = np.where((lon5_i >= LON0) & (lon5_i <= LON1))[0]
row_map = {r: i for i, r in enumerate(R5)}
col_map = {c: j for j, c in enumerate(C5)}
CELL5 = (lat5_i[R5] - LAT0)[:, None] * NLON + (lon5_i[C5] - LON0)[None, :]
AREA5 = np.load(CELL_AREA_FILE)[np.ix_(R5, C5)]

land_rg = regionmask.defined_regions.natural_earth_v5_0_0.land_10
land5 = np.isfinite(land_rg.mask(lon5[C5], lat5[R5]).values)
land_frac = np.bincount(CELL5.ravel(), weights=land5.ravel().astype(float), minlength=NCELL) / 144.0
LAND1 = land_frac >= LAND1_THR
keep5 = land5 & LAND1[CELL5]
# surface de TERRE totale (ha) par cellule -- somme de AREA5 sur les pixels natifs geographiquement
# "terre" ET dans une cellule 1deg classee "terre" (meme convention que keep5 partout ailleurs)
land_area_cell = np.bincount(CELL5.ravel(), weights=np.where(keep5, AREA5, 0).ravel(), minlength=NCELL).reshape(NLAT, NLON)

CAL_CLASSES = {"Wheat": (1, 27), "Maize": (2, 28), "Potato": (10, 36)}
cols = ["cell", "row", "col", "lat", "lon", "crop", "subcrop", "area", "start", "end"]
cal = pd.read_csv(CAL_FILE, sep="\t", compression="gzip", usecols=range(10), header=0, names=cols)
allc = [c for v in CAL_CLASSES.values() for c in v]
cal = cal[cal.crop.isin(allc)].copy()
cal["i"] = (cal.row - 1).map(row_map)
cal["j"] = (cal.col - 1).map(col_map)
cal = cal.dropna(subset=["i", "j"])
cal["i"] = cal.i.astype(int); cal["j"] = cal.j.astype(int)
cal["irr"] = np.where(cal.crop <= 26, "ir", "rf")
name_of = {c: k for k, v in CAL_CLASSES.items() for c in v}
cal["crop_name"] = cal.crop.map(name_of)
print("calendar sub-crop rows in window:", len(cal))


def mirca2000_presence_1deg_cellareafrac(crop_name, irr, season=None):
    g = cal[(cal.crop_name == crop_name) & (cal.irr == irr)]
    if season == 'winter':
        g = g[g.start.isin([9, 10, 11, 12])]
    elif season == 'spring':
        g = g[g.start.isin([2, 3, 4, 5])]
    # Pixel natif : AUCUN seuil, toute trace de surface compte (pas de filtre de concentration locale)
    cell_idx = CELL5[g.i.values, g.j.values]
    crop_area_cell = np.bincount(cell_idx, weights=g.area.values, minlength=NCELL).reshape(NLAT, NLON)
    # Cellule 1deg : seuil sur la fraction de SURFACE reelle (pas un comptage de pixels)
    cell_area_frac = np.where(land_area_cell > 0, crop_area_cell / np.maximum(land_area_cell, 1), 0.0)
    pres = (land_area_cell > 0) & (cell_area_frac > CELL_AREA_THR)
    return np.where(land_mask, pres.astype(float), np.nan), np.where(land_mask, crop_area_cell, np.nan)


presence_by_label = {}
for crop_label, (crop_name, irr, season, _old) in CROP_MAP.items():
    pres, area_cell = mirca2000_presence_1deg_cellareafrac(crop_name, irr, season)
    presence_by_label[crop_label] = pres
    n_pres = int(np.nansum(pres == 1)); n_land = int(np.sum(land_mask))
    print(f"{crop_label:18s}: presence (pixel sans seuil, cellule surface>{CELL_AREA_THR*100:.1f}%) = "
          f"{n_pres} / {n_land} land cells ({100*n_pres/n_land:.1f}%)")


calendar sub-crop rows in window: 591065


Maize_rf          : presence (pixel sans seuil, cellule surface>0.1%) = 870 / 1332 land cells (65.3%)
Wheat-spring_rf   : presence (pixel sans seuil, cellule surface>0.1%) = 516 / 1332 land cells (38.7%)


Wheat-winter_rf   : presence (pixel sans seuil, cellule surface>0.1%) = 983 / 1332 land cells (73.8%)
Potato_rf         : presence (pixel sans seuil, cellule surface>0.1%) = 867 / 1332 land cells (65.1%)


## 3. Sauvegarde -- reutilise le calendrier (sowing/maturity DOY) de MIRCA2000 original, inchange

In [4]:
for crop_label, (crop_name, irr, season, old_file) in CROP_MAP.items():
    old_ds = xr.open_dataset(ERA5_1DEG / old_file)
    sowing_doy = old_ds['sowing_doy'].values
    maturity_doy = old_ds['maturity_doy'].values
    lats_ref = old_ds.lat.values; lons_ref = old_ds.lon.values
    old_ds.close()

    new_presence = presence_by_label[crop_label]
    out = xr.Dataset({
        'presence': xr.DataArray(new_presence, dims=['lat', 'lon'], coords={'lat': lats_ref, 'lon': lons_ref},
                                 attrs={'long_name': f'MIRCA2000 presence, pixel sans seuil, cellule surface>{CELL_AREA_THR*100:.1f}%, crop={crop_label}'}),
        'sowing_doy': xr.DataArray(sowing_doy, dims=['lat', 'lon'], coords={'lat': lats_ref, 'lon': lons_ref},
                                   attrs={'long_name': 'Sowing DOY (unchanged, from MIRCA2000 5%-threshold calendar)'}),
        'maturity_doy': xr.DataArray(maturity_doy, dims=['lat', 'lon'], coords={'lat': lats_ref, 'lon': lons_ref},
                                     attrs={'long_name': 'Maturity DOY (unchanged, from MIRCA2000 5%-threshold calendar)'}),
    })
    out.attrs['description'] = (f'Presence from MIRCA2000, pixel sans seuil, cellule surface>{CELL_AREA_THR} (agronomic choice) -- {crop_label}')
    out_path = ERA5_1DEG / f'mirca2000_cellareafrac_1deg_{crop_label}.nc'
    out.to_netcdf(out_path)
    print(f"Saved: {out_path.name}  (presence cells: {int(np.nansum(new_presence==1))})")


Saved: mirca2000_cellareafrac_1deg_Maize_rf.nc  (presence cells: 870)


Saved: mirca2000_cellareafrac_1deg_Wheat-spring_rf.nc  (presence cells: 516)


Saved: mirca2000_cellareafrac_1deg_Wheat-winter_rf.nc  (presence cells: 983)


Saved: mirca2000_cellareafrac_1deg_Potato_rf.nc  (presence cells: 867)
